# TextGuard — audit your own text-fed forecaster

Measures how far misleading report text can push **your** forecaster (DSS, flip rate), and what
TextGuard v1 / v2, clipping and never_trust cost on genuine reports (False Block Rate, MSE).
Paper code and settings: [https://github.com/aryanwaingankar4/textguard](https://github.com/aryanwaingankar4/textguard).

* **Runtime:** a GPU (Colab: Runtime → Change runtime type → T4 GPU; Kaggle: Accelerator GPU T4,
  Internet on). The judge is Qwen2.5-1.5B-Instruct; on a CPU it is ~50× slower.
* **Worked example:** runs as is (a dummy model on a small synthetic series). For your model, edit
  cells 3 and 4 only.
* **Honest limits** (README "Use TextGuard on your own model"): the judge reads a direction into
  about 1 in 3 neutral financial headlines, so v2 can block many genuine headline bundles (18–22% in
  our held-out test); neither version helped Qwen2.5-7B there; v1 is evaded by an appended
  forward-looking sentence. Treat the audit as a measurement, not a guarantee.
* **Ethics:** the attack texts this notebook writes are synthetic, for testing defences only.

In [ ]:
# 1. Get the code (needs the repository to be reachable; otherwise upload a zip and unzip it here)
import os, subprocess, sys
if not os.path.exists("textguard"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/aryanwaingankar4/textguard"], check=True)
os.chdir("textguard")
# Colab / Kaggle already ship torch, transformers, pandas and numpy; only add what is missing
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "pyarrow"], check=True)

In [ ]:
# 2. Imports and the GPU check
import sys, torch
sys.path.insert(0, ".")
from textguard import DummyForecaster, audit, judges, load_series
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: the judge will be slow)")

In [ ]:
# 3. YOUR DATA — a CSV with columns date, value, text (+ split or TRAIN_END; optional fake_text,
#    fake_direction). `text` = the report(s) available BEFORE this row's value is known.
DATA = "examples/sample_series.csv"      # or upload: from google.colab import files; files.upload()
LOOKBACK = 12                            # values the model sees per forecast
FREQ = "weekly"                          # "weekly" / "monthly" (sets v2's year-over-year lag) or None
SUBJECT = "the average widget price"     # what the series is, in words (the judge's question)
FRAME_V1 = "compared with the previous week"   # v1 only: the frame of a typical claim ("" if none)
KEY = "my-series"                        # any name for this series
TRAIN_END = None                         # e.g. "2023-06-30" if the CSV has no `split` column
data = load_series(DATA, lookback=LOOKBACK, freq=FREQ, train_end=TRAIN_END)
print(len(data.windows), "evaluation windows; sigma (train) =", data.sigma)

In [ ]:
# 4. YOUR MODEL — any object with `horizon` and `predict(numbers, text) -> array of shape (horizon,)`;
#    numbers = the last LOOKBACK values (raw units), text = str or None (None = numbers only).
model = DummyForecaster(sensitivity=1.0, step=data.sigma / 4, lookback=LOOKBACK)
# class MyModel:
#     horizon = 1
#     def predict(self, numbers, text):
#         ...                      # call your forecaster here
# model = MyModel()

In [ ]:
# 5. The judges (Qwen2.5-1.5B-Instruct @ 989aa79, fp16 on the GPU; one model for v1 and v2)
judge_v1, judge_v2 = judges(KEY, SUBJECT, frame_v1=FRAME_V1, device=DEVICE)

In [ ]:
# 6. The audit: undefended, never_trust, clip (k 0.5), TextGuard v1 and v2 (frozen paper settings)
report = audit(model, data, judge_v1=judge_v1, judge_v2=judge_v2, subject=SUBJECT)
report.save("audit_out")
judge_v1.scorer.answers().to_csv("audit_out/judge_answers.csv", index=False)
try:
    from IPython.display import Markdown, display
    display(Markdown(report.to_markdown()))
except ImportError:
    print(report.to_markdown())

In [ ]:
# 7. Download the results
import shutil
shutil.make_archive("audit_out", "zip", "audit_out")
try:
    from google.colab import files
    files.download("audit_out.zip")
except ImportError:
    print("Kaggle / local: audit_out.zip is in", os.getcwd())